In [ ]:
import sys, os

# Remonter jusqu'à la racine du dépôt (2 niveaux au-dessus de notebooks/tachyon_DE/)
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"Racine du dépôt : {repo_root}")
print(f"sys.path[0]     : {sys.path[0]}")

import importlib.util
spec = importlib.util.find_spec("src.tachyon.potentials")
print(f"src.tachyon.potentials trouvé : {spec is not None}")

In [1]:
import os

packages_dir = os.path.expanduser("~/cobaya_packages")
for root, dirs, files in os.walk(packages_dir):
    depth = root.replace(packages_dir, '').count(os.sep)
    if depth <= 2:
        print('  '*depth + os.path.basename(root) + '/')

cobaya_packages/
  code/
    planck/
  data/
    planck_2018_lowE_native/
    bao_data/
    planck_supp_data_and_covmats/
    planck_2018/
    planck_2018_lowT_native/
    sn_data/


In [2]:
import glob
bao_files = glob.glob(os.path.join(packages_dir, "**/*desi*dr2*"), recursive=True)
for f in bao_files[:20]:
    print(f)

/Users/ana.feralvar/cobaya_packages/data/bao_data/desi_bao_dr2


## Comment cobaya reçoit un modèle de fond personnalisé

Pour injecter notre tachyon, deux approches possibles :

1. **`classy`/`camb` avec `dark_energy_model: 'w0wa'` paramétré point
   par point** — on fournit w(z) tabulé, CAMB interpole (PPF).
2. **Une classe `Theory` cobaya personnalisée** — on calcule nous-mêmes
   H(z), D_M(z), D_H(z) à partir de `rolling_tachyon.py` et on les
   fournit directement aux likelihoods BAO/SNe (sans passer par CAMB
   pour le fond tardif). Le CMB (Planck) reste calculé par CAMB avec
   un w(z) tabulé en entrée.

**Option 2 est plus simple et plus robuste** pour notre cas : on
n'a pas besoin du detail de la physique des perturbations du
tachyon, seulement de son effet sur le fond H(z).

In [3]:
# Vérifier que CAMB accepte un w(z) tabulé (PPF dark energy)
import camb
help(camb.CAMBparams.set_dark_energy)

Help on function set_dark_energy in module camb.model:

set_dark_energy(self, w=-1.0, cs2=1.0, wa=0, dark_energy_model='fluid')
    Set dark energy parameters (use set_dark_energy_w_a to set w(a) from numerical table instead)
    To use a custom dark energy model, assign the class instance to the DarkEnergy field instead.
    
    :param w: :math:`w\equiv p_{\rm de}/\rho_{\rm de}`, assumed constant
    :param wa: evolution of w (for dark_energy_model=ppf)
    :param cs2: rest-frame sound speed squared of dark energy fluid
    :param dark_energy_model: model to use ('fluid' or 'ppf'), default is 'fluid'
    :return: self



In [ ]:
import camb
import numpy as np

pars = camb.CAMBparams()
pars.set_cosmology(H0=67.5, ombh2=0.0224, omch2=0.120)

# w(z) tabulé depuis notre tachyon
from src.tachyon.potentials import ExponentialPotential
from src.tachyon.rolling_tachyon import build_tachyon_background

pot = ExponentialPotential(V0=1.0, lam=0.7)
bg = build_tachyon_background(pot, Omega_m=0.30, phi_init=0.0,
                                x_init=0.0, z_init=50.0)

a_arr = bg["a"][::-1]  # croissant
w_arr = bg["w"][::-1]

pars.set_dark_energy(w=-1.0, dark_energy_model='ppf')  # placeholder
# -> on creusera l'API exacte pour w(a) tabulé (DarkEnergyFluid /
#    DarkEnergyPPF avec set_w_a_table)

print("a range:", a_arr.min(), a_arr.max())
print("w range:", w_arr.min(), w_arr.max())

ModuleNotFoundError: No module named 'src'

In [5]:
import os, glob

base = os.path.expanduser("~/cobaya_packages/data")

for folder in sorted(os.listdir(base)):
    full = os.path.join(base, folder)
    files = os.listdir(full)
    print(f"\n📁 {folder}/  ({len(files)} fichiers)")
    for f in sorted(files)[:15]:
        size = os.path.getsize(os.path.join(full, f)) / 1024
        print(f"   {f:<45} {size:>8.1f} KB")


📁 bao_data/  (45 fichiers)
   .gitignore                                         0.0 KB
   BAO_consensus_covtot_dM_Hz.txt                     0.3 KB
   FS_consensus_covtot_dM_Hz_fsig.txt                 0.7 KB
   README.md                                          1.2 KB
   desi_2024_eboss_gaussian_bao_Lya_GCcomb_cov.txt      0.1 KB
   desi_2024_eboss_gaussian_bao_Lya_GCcomb_mean.txt      0.1 KB
   desi_2024_gaussian_bao_ALL_GCcomb_cov.txt          2.1 KB
   desi_2024_gaussian_bao_ALL_GCcomb_mean.txt         0.4 KB
   desi_2024_gaussian_bao_BGS_BRIGHT-21.5_GCcomb_z0.1-0.4_cov.txt      0.0 KB
   desi_2024_gaussian_bao_BGS_BRIGHT-21.5_GCcomb_z0.1-0.4_mean.txt      0.1 KB
   desi_2024_gaussian_bao_ELG_LOPnotqso_GCcomb_z1.1-1.6_cov.txt      0.1 KB
   desi_2024_gaussian_bao_ELG_LOPnotqso_GCcomb_z1.1-1.6_mean.txt      0.1 KB
   desi_2024_gaussian_bao_LRG+ELG_LOPnotqso_GCcomb_z0.8-1.1_cov.txt      0.1 KB
   desi_2024_gaussian_bao_LRG+ELG_LOPnotqso_GCcomb_z0.8-1.1_mean.txt      0.1 KB
   desi_

In [6]:
bao_path = os.path.expanduser("~/cobaya_packages/data/bao_data")
print("Fichiers BAO :")
for root, dirs, files in os.walk(bao_path):
    for f in sorted(files):
        full = os.path.join(root, f)
        size = os.path.getsize(full) / 1024
        rel = os.path.relpath(full, bao_path)
        print(f"  {rel:<50} {size:>8.1f} KB")

Fichiers BAO :
  .gitignore                                              0.0 KB
  BAO_consensus_covtot_dM_Hz.txt                          0.3 KB
  FS_consensus_covtot_dM_Hz_fsig.txt                      0.7 KB
  README.md                                               1.2 KB
  desi_2024_eboss_gaussian_bao_Lya_GCcomb_cov.txt         0.1 KB
  desi_2024_eboss_gaussian_bao_Lya_GCcomb_mean.txt        0.1 KB
  desi_2024_gaussian_bao_ALL_GCcomb_cov.txt               2.1 KB
  desi_2024_gaussian_bao_ALL_GCcomb_mean.txt              0.4 KB
  desi_2024_gaussian_bao_BGS_BRIGHT-21.5_GCcomb_z0.1-0.4_cov.txt      0.0 KB
  desi_2024_gaussian_bao_BGS_BRIGHT-21.5_GCcomb_z0.1-0.4_mean.txt      0.1 KB
  desi_2024_gaussian_bao_ELG_LOPnotqso_GCcomb_z1.1-1.6_cov.txt      0.1 KB
  desi_2024_gaussian_bao_ELG_LOPnotqso_GCcomb_z1.1-1.6_mean.txt      0.1 KB
  desi_2024_gaussian_bao_LRG+ELG_LOPnotqso_GCcomb_z0.8-1.1_cov.txt      0.1 KB
  desi_2024_gaussian_bao_LRG+ELG_LOPnotqso_GCcomb_z0.8-1.1_mean.txt      0.1 KB


In [7]:
sn_path = os.path.expanduser("~/cobaya_packages/data/sn_data")
print("Fichiers SNe :")
for root, dirs, files in os.walk(sn_path):
    for f in sorted(files):
        full = os.path.join(root, f)
        size = os.path.getsize(full) / 1024
        rel = os.path.relpath(full, sn_path)
        print(f"  {rel:<50} {size:>8.1f} KB")

Fichiers SNe :
  .gitignore                                              0.0 KB
  README.md                                               1.2 KB
  version.dat                                             0.0 KB
  DES-Dovekie/DES-Dovekie_HD.csv                        106.0 KB
  DES-Dovekie/config.dataset                              0.1 KB
  DES-Dovekie/covtot_inv_000.npz                       6098.6 KB
  JLA/jla.dataset                                         0.7 KB
  JLA/jla_lcparams.txt                                  101.2 KB
  JLA/jla_v0_covmatrix.dat                             9326.1 KB
  JLA/jla_v0a_covmatrix.dat                            9796.1 KB
  JLA/jla_v0b_covmatrix.dat                            9570.2 KB
  JLA/jla_va_covmatrix.dat                             9747.0 KB
  JLA/jla_vab_covmatrix.dat                            9852.5 KB
  JLA/jla_vb_covmatrix.dat                             9627.7 KB
  PantheonPlus/Pantheon+SH0ES.dat                       565.7 KB
  Pantheon

In [8]:
from cobaya.likelihoods import bao, sn
print("BAO likelihoods disponibles :")
import cobaya.likelihoods.bao as bao_mod
print(dir(bao_mod))

print("\nSN likelihoods disponibles :")
import cobaya.likelihoods.sn as sn_mod
print(dir(sn_mod))

BAO likelihoods disponibles :
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']

SN likelihoods disponibles :
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


In [ ]:
from cobaya.model import get_model

info_test = {
    "likelihood": {
        "bao.desi_dr2": None
    },
    "theory": {
        "camb": {"extra_args": {"num_massive_neutrinos": 1}}
    },
    "params": {
        "H0": 67.5,
        "ombh2": 0.0224,
        "omch2": 0.120,
        "As": 2.1e-9,
        "ns": 0.965,
        "tau": 0.055,
    },
    "packages_path": os.path.expanduser("~/cobaya_packages")
}

try:
    model = get_model(info_test)
    lp = model.loglikes({})[0]
    print(f"Log-likelihood BAO DESI DR2 = {lp}")
    print("Likelihood BAO fonctionne !")
except Exception as e:
    print(f"Erreur : {e}")

[camb] `camb` module loaded successfully from /opt/anaconda3/envs/souriau/lib/python3.11/site-packages/camb
[bao.desi_dr2] Initialized.
Log-likelihood BAO DESI DR2 = [-13.39689479]
Likelihood BAO fonctionne !


In [ ]:
import camb

# Comment passer w(z) tabulé à CAMB?
help(camb.dark_energy.DarkEnergyFluid)

Help on class DarkEnergyFluid in module camb.dark_energy:

class DarkEnergyFluid(DarkEnergyEqnOfState)
 |  DarkEnergyFluid(*args, **kwargs)
 |  
 |  Class implementing the w, wa or splined w(a) parameterization using the constant sound-speed single fluid model
 |  (as for single-field quintessence).
 |  
 |  Method resolution order:
 |      DarkEnergyFluid
 |      DarkEnergyEqnOfState
 |      DarkEnergyModel
 |      camb.baseconfig.F2003Class
 |      camb.baseconfig.CAMB_Structure
 |      _ctypes.Structure
 |      _ctypes._CData
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  set_w_a_table(self, a, w) -> 'DarkEnergyEqnOfState'
 |      Set w(a) from numerical values (used as cubic spline). Note this is quite slow.
 |      
 |      :param a: array of scale factors
 |      :param w: array of w(a)
 |      :return: self
 |  
 |  validate_params(self) -> None
 |  
 |  ----------------------------------------------------------------------
 |  Data and other attributes defined 